In [2]:
import mediapipe as mp
import cv2 as cv 
import numpy as np 

In [3]:
mp_face_mesh = mp.solutions.face_mesh 
face_mesh = mp_face_mesh.FaceMesh()

In [4]:
UPPER_LIP = [13, 61, 291, 185, 40, 39, 37, 267, 269, 270, 409]
LOWER_LIP = [146, 91, 181, 84, 17, 314, 405, 321, 375]

def apply_lipstick(img, landmarks, color=(0, 0, 255)):
    points = []
    for idx in  UPPER_LIP + LOWER_LIP:
        x, y = int(landmarks[idx].x * img.shape[1]), int(landmarks[idx].y * img.shape[0])
        points.append((x, y))

    points = np.array(points, np.int32)
    cv.fillPoly(img, [points], color)
    return img

In [5]:
def apply_skin_smooth(img):
    smooth = cv.GaussianBlur(img, (15, 15), 30)
    return cv.addWeighted(img, 0.7, smooth, 0.3, 0)

In [6]:
def apply_blush(image, landmarks, color=(255, 0, 0)):
    h, w, _ = image.shape
    cheek_indices = [116, 117, 118, 119, 120, 121, 350, 349, 348, 347, 346, 345]
    cheek_points = []
    for idx in cheek_indices:
        landmark = landmarks.landmark[idx]
        x, y = int(landmark.x * w), int(landmark.y * h)
        cheek_points.append((x, y))
    # Create a mask for the cheeks
    mask = cv.fillPoly(np.zeros_like(image), [np.array(cheek_points)], color)
    # Blend the mask with the original image
    image = cv.addWeighted(image, 1, mask, 0.3, 0)
    return image

In [7]:
HAIRLINE_INDICES = [10, 67, 69, 108, 109, 151, 299, 337, 332]
def change_hair_color(img, landmarks, color=(255, 0, 0)):
    points = []
    for idx in HAIRLINE_INDICES:
        x, y = int(landmarks[idx].x * img.shape[1]), int(landmarks[idx].y * img.shape[0])
        points.append((x, y))
    points = np.array(points, np.int32)
    mask = np.zeros_like(img)
    cv.fillPoly(mask, [points], color)
    return cv.addWeighted(img, 0.7, mask, 0.3, 0)

In [8]:
def apply_eyeliner(image, landmarks, color=(0, 0, 0)):  # Black color
    h, w, _ = image.shape
    eyeliner_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
    eyeliner_points = []
    for idx in eyeliner_indices:
        landmark = landmarks.landmark[idx]
        x, y = int(landmark.x * w), int(landmark.y * h)
        eyeliner_points.append((x, y))
    
    # Draw eyeliner using lines
    for i in range(len(eyeliner_points) - 1):
        cv.line(image, eyeliner_points[i], eyeliner_points[i + 1], color, 2)
    return image

In [9]:
def apply_contouring(image, landmarks, color=(0, 0, 0)): 
    h, w, _ = image.shape
    contour_indices = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377]
    contour_points = []
    for idx in contour_indices:
        landmark = landmarks.landmark[idx]
        x, y = int(landmark.x * w), int(landmark.y * h)
        contour_points.append((x, y))
    
    # Create a mask for contouring
    mask = cv.fillPoly(np.zeros_like(image), [np.array(contour_points)], color)
    # Blend the mask with the original image
    image = cv.addWeighted(image, 1, mask, 0.2, 0)
    return image

In [10]:
cam = cv.VideoCapture(0)
while True:
    ret,img = cam.read()
    if not ret:
        break 
    rgb_img = cv.cvtColor(img,cv.COLOR_BGR2RGB)
    process = face_mesh.process(rgb_img)
    if process.multi_face_landmarks:
        for facelandmarks in process.multi_face_landmarks:
            img = apply_lipstick(img,facelandmarks.landmark)
            img = apply_blush(img,facelandmarks)
            img = apply_eyeliner(img,facelandmarks)
            img = apply_contouring(img,facelandmarks)
            img = apply_skin_smooth(img)
                
    cv.imshow("facemesh",img)
    if cv.waitKey(1) & 0xff == ord('r'):
        break
cam.release()
cv.destroyAllWindows()

In [ ]:
img_path = "1.jpg"

img = cv.imread(img_path)
rgb_img = cv.cvtColor(img,cv.COLOR_BGR2RGB)
process = face_mesh.process(rgb_img)

if process.multi_face_landmarks:
    for facelandmarks in process.multi_face_landmarks:
        img = apply_lipstick(img,facelandmarks.landmark)
        img = apply_blush(img,facelandmarks)
        img = apply_eyeliner(img,facelandmarks)
        img = apply_contouring(img,facelandmarks)
        img = apply_skin_smooth(img)
        
cv.imshow("img",img)
cv.waitKey(0)

27

Exception in callback BaseSelectorEventLoop._read_from_self()
handle: <Handle BaseSelectorEventLoop._read_from_self()>
Traceback (most recent call last):
  File "c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\asyncio\selector_events.py", line 119, in _read_from_self
    data = self._ssock.recv(4096)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback BaseSelectorEventLoop._read_from_self()
handle: <Handle BaseSelectorEventLoop._read_from_self()>
Traceback (most recent call last):
  File "c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python37\lib\asyncio\selector_events.py", line 119, 

: 

In [ ]:
import cv2 

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
cap = cv.VideoCapture(0)
with mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    while cap.isOpened():
        ret, img = cap.read()
        if not ret:
            break
        
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        
        results = face_mesh.process(rgb_img)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                for i, landmark in enumerate(face_landmarks.landmark):
                    h, w, _ = img.shape
                    x, y = int(landmark.x * w), int(landmark.y * h)

                    
                    text = f"{i}: ({x}, {y})"
                    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)

       
        cv2.imshow("Face Mesh with Coordinates", img)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [38]:
import cv2
import mediapipe as mp

# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh

# Define specific landmark indices to display
selected_landmarks = {
    1: "Nose", 
    33: "Left Eye", 
    263: "Right Eye", 
    13: "Upper Lip", 
    14: "Lower Lip"
}

# Start webcam capture
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    while cap.isOpened():
        ret, img = cap.read()
        if not ret:
            break

        # Convert to RGB (MediaPipe requires RGB format)
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Process the img with Face Mesh
        results = face_mesh.process(rgb_img)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                for idx, name in selected_landmarks.items():
                    landmark = face_landmarks.landmark[idx]
                    h, w, _ = img.shape
                    x, y = int(landmark.x * w), int(landmark.y * h)

                    # Draw a small circle on the landmark
                    cv2.circle(img, (x, y), 3, (0, 255, 0), -1)

                    # Display landmark name and coordinates
                    text = f"{name}: ({x}, {y})"
                    cv2.putText(img, text, (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        # Show the output
        cv2.imshow("Face Mesh with Selected Coordinates", img)

        # Press 'q' to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
